In [55]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from configs import config

In [47]:
df = spark.read.csv(
    "data/raw/Attraction_Belem.csv",
    header=True,
    sep=",",
    quote='"',
    escape='"',
    multiLine=True
)
df.show(5)

+---+-----------------+----------------+----------------+--------------------+-------------+--------------------+------------------+-----------------+-------------+
|_c0|             Name|Rating_attaction|        Username|        Title_Review|Rating_Review|              Review|              Date|            State|Type_traveler|
+---+-----------------+----------------+----------------+--------------------+-------------+--------------------+------------------+-----------------+-------------+
|  0|Estação das Docas|              45| Volnei Castanho|    Local agradável!|           50|Local reformado, ...| fevereiro de 2020|      Barueri, SP|     Families|
|  1|Estação das Docas|              45|Alexandre Mendes|      Lugar incrível|           50|Lugar incrível pa...|   janeiro de 2020|        Belém, PA|     Families|
|  2|Estação das Docas|              45|        Airton C|Muitas opções em ...|           50|O espaço é muito ...|     março de 2020|   Ouro Preto, MG|     Families|
|  3|Estaç

In [30]:
df.summary().show()

+-------+------------------+--------------------+------------------+----------+--------------------+-----------------+--------------------+-----------------+---------------+-------------+
|summary|               _c0|                Name|  Rating_attaction|  Username|        Title_Review|    Rating_Review|              Review|             Date|          State|Type_traveler|
+-------+------------------+--------------------+------------------+----------+--------------------+-----------------+--------------------+-----------------+---------------+-------------+
|  count|             14098|               14098|             14098|     14098|               14098|            14098|               14098|            14098|          11884|        14098|
|   mean|            7048.5|                NULL| 45.38196907362747|      NULL|        2.90024243E8|45.69938998439495|                NULL|             NULL|           NULL|         NULL|
| stddev|4069.8863825255203|                NULL|2.147601880

In [31]:
df.dtypes

[('_c0', 'string'),
 ('Name', 'string'),
 ('Rating_attaction', 'string'),
 ('Username', 'string'),
 ('Title_Review', 'string'),
 ('Rating_Review', 'string'),
 ('Review', 'string'),
 ('Date', 'string'),
 ('State', 'string'),
 ('Type_traveler', 'string')]

In [32]:
df.printSchema()

root
 |-- _c0: string (nullable = true)
 |-- Name: string (nullable = true)
 |-- Rating_attaction: string (nullable = true)
 |-- Username: string (nullable = true)
 |-- Title_Review: string (nullable = true)
 |-- Rating_Review: string (nullable = true)
 |-- Review: string (nullable = true)
 |-- Date: string (nullable = true)
 |-- State: string (nullable = true)
 |-- Type_traveler: string (nullable = true)



In [33]:
df = df.drop('_c0')

In [34]:
df_bronze = df.filter((F.col("Name").isNotNull()) & (F.trim(F.col("Name")) != ""))
df_bronze.show(5)

+-----------------+----------------+----------------+--------------------+-------------+--------------------+------------------+-----------------+-------------+
|             Name|Rating_attaction|        Username|        Title_Review|Rating_Review|              Review|              Date|            State|Type_traveler|
+-----------------+----------------+----------------+--------------------+-------------+--------------------+------------------+-----------------+-------------+
|Estação das Docas|              45| Volnei Castanho|    Local agradável!|           50|Local reformado, ...| fevereiro de 2020|      Barueri, SP|     Families|
|Estação das Docas|              45|Alexandre Mendes|      Lugar incrível|           50|Lugar incrível pa...|   janeiro de 2020|        Belém, PA|     Families|
|Estação das Docas|              45|        Airton C|Muitas opções em ...|           50|O espaço é muito ...|     março de 2020|   Ouro Preto, MG|     Families|
|Estação das Docas|              4

In [35]:
df_bronze = df_bronze.withColumn(
    "Date_Travel",
    F.to_date(
        F.concat_ws(
            "-",
            F.regexp_extract(F.col("Date"), r"(\d{4})", 1),
            F.lpad(
                F.when(F.col("Date").rlike("janeiro"), "01")
                 .when(F.col("Date").rlike("fevereiro"), "02")
                 .when(F.col("Date").rlike("março|marco"), "03")
                 .when(F.col("Date").rlike("abril"), "04")
                 .when(F.col("Date").rlike("maio"), "05")
                 .when(F.col("Date").rlike("junho"), "06")
                 .when(F.col("Date").rlike("julho"), "07")
                 .when(F.col("Date").rlike("agosto"), "08")
                 .when(F.col("Date").rlike("setembro"), "09")
                 .when(F.col("Date").rlike("outubro"), "10")
                 .when(F.col("Date").rlike("novembro"), "11")
                 .when(F.col("Date").rlike("dezembro"), "12")
                 .otherwise(None),
                2,
                "0"
            ),
            F.lit("01")
        ),
        "yyyy-MM-dd"
    )
)
df_bronze = df_bronze.withColumn("Rating_Review", F.col("Rating_Review").cast("int"))
df_bronze = df_bronze.withColumn("Rating_attaction", F.col("Rating_attaction").cast("int"))
df_bronze.show(5)

+-----------------+----------------+----------------+--------------------+-------------+--------------------+------------------+-----------------+-------------+-----------+
|             Name|Rating_attaction|        Username|        Title_Review|Rating_Review|              Review|              Date|            State|Type_traveler|Date_Travel|
+-----------------+----------------+----------------+--------------------+-------------+--------------------+------------------+-----------------+-------------+-----------+
|Estação das Docas|              45| Volnei Castanho|    Local agradável!|           50|Local reformado, ...| fevereiro de 2020|      Barueri, SP|     Families| 2020-02-01|
|Estação das Docas|              45|Alexandre Mendes|      Lugar incrível|           50|Lugar incrível pa...|   janeiro de 2020|        Belém, PA|     Families| 2020-01-01|
|Estação das Docas|              45|        Airton C|Muitas opções em ...|           50|O espaço é muito ...|     março de 2020|   Ouro

In [36]:
df_bronze.show(5)

+-----------------+----------------+----------------+--------------------+-------------+--------------------+------------------+-----------------+-------------+-----------+
|             Name|Rating_attaction|        Username|        Title_Review|Rating_Review|              Review|              Date|            State|Type_traveler|Date_Travel|
+-----------------+----------------+----------------+--------------------+-------------+--------------------+------------------+-----------------+-------------+-----------+
|Estação das Docas|              45| Volnei Castanho|    Local agradável!|           50|Local reformado, ...| fevereiro de 2020|      Barueri, SP|     Families| 2020-02-01|
|Estação das Docas|              45|Alexandre Mendes|      Lugar incrível|           50|Lugar incrível pa...|   janeiro de 2020|        Belém, PA|     Families| 2020-01-01|
|Estação das Docas|              45|        Airton C|Muitas opções em ...|           50|O espaço é muito ...|     março de 2020|   Ouro

In [37]:
#--------------Modelagem-------------
# Attraction -> ID, Name, Rating_attraction
### User -> ID, Username, State (Cardinality problem)
# User_Review -> ID, ID_User, ID_Attraction, Title_Review, Rating_Review, Review, Username, State, Type_traveler, Date

In [38]:
# dim_name
df_dim_silver_attractions = (
    df_bronze.select(["Name","Rating_attaction"])
      .where(F.trim(F.col("Name")).isNotNull())
      .dropDuplicates(["Name"])
      .withColumn("Attraction_id", F.row_number().over(Window.orderBy("Name")))
      .withColumn("Rating_Attraction", F.col("Rating_attaction"))
      .select("Attraction_id", "Name", "Rating_Attraction")
)
df_dim_silver_attractions.show(100,truncate=False)

+-------------+---------------------------------------+-----------------+
|Attraction_id|Name                                   |Rating_Attraction|
+-------------+---------------------------------------+-----------------+
|1            |Basílica de Nossa Senhora de Nazaré    |50               |
|2            |Estação das Docas                      |45               |
|3            |Forte do Presépio                      |45               |
|4            |Hangar Centro de Convenções da Amazônia|45               |
|5            |Mangal das Garças                      |45               |
|6            |Museu Paraense Emílio Goeldi           |45               |
|7            |Parque Estadual do Utinga              |45               |
|8            |Parque da Residência                   |45               |
|9            |Praça da República                     |40               |
|10           |Teatro da Paz                          |45               |
+-------------+-----------------------

In [39]:
df_fact_silver_reviews = df_bronze.drop("Rating_attaction", "Date")

fact = df_fact_silver_reviews.alias("fact")
dim = df_dim_silver_attractions.alias("dim")

df_fact_silver_reviews = (
    fact.join(
        dim,
        F.lower(F.col("fact.Name")) == F.lower(F.col("dim.Name")),
        "left"
    )
    .drop("Name")           
    .drop("Rating_Attraction") 
)

df_fact_silver_reviews.show(5)


+----------------+--------------------+-------------+--------------------+-----------------+-------------+-----------+-------------+
|        Username|        Title_Review|Rating_Review|              Review|            State|Type_traveler|Date_Travel|Attraction_id|
+----------------+--------------------+-------------+--------------------+-----------------+-------------+-----------+-------------+
| Volnei Castanho|    Local agradável!|           50|Local reformado, ...|      Barueri, SP|     Families| 2020-02-01|            2|
|Alexandre Mendes|      Lugar incrível|           50|Lugar incrível pa...|        Belém, PA|     Families| 2020-01-01|            2|
|        Airton C|Muitas opções em ...|           50|O espaço é muito ...|   Ouro Preto, MG|     Families| 2020-03-01|            2|
|        marcos l|         Restaurante|           50|Um ótimo lugar pa...|        Belém, PA|     Families| 2020-02-01|            2|
|  Ana Caroline T|                 Top|           50|Vale demais a vi

# Quality Tests dim attractions

In [40]:
df_validation_dim_attractions = df_dim_silver_attractions.withColumn("validation_errors", F.lit(""))

# Rule 1: 'id' is REQUIRED and should not be null

df_validation_dim_attractions = df_validation_dim_attractions.withColumn(
    "validation_errors",
    F.when(F.isnull(F.col("Attraction_id")), F.concat(F.col("validation_errors"), F.lit("ID_NULL;"))) # Changed to use concat function
    .otherwise(F.col("validation_errors"))
)

# Rule 2: 'id' must be unique

w = Window.partitionBy("Attraction_id")

df_validation_dim_attractions = df_validation_dim_attractions.withColumn(
    "validation_errors",
    F.when(
        F.count("Attraction_id").over(w) > 1,
        F.concat(F.col("validation_errors"), F.lit("ID_NOT_UNIQUE"))
    ).otherwise(F.col("validation_errors"))
)

 # Rule 3: 'Rating_Attraction' should be a positive integer and less than 50

df_validation_dim_attractions = df_validation_dim_attractions.withColumn(
"validation_errors",
F.when((F.col("Rating_Attraction") <= 0) | (F.col("Rating_Attraction") > 50), F.concat(F.col("validation_errors"), F.lit("RATING_ATTRACTION_INVALID;"))) # Changed to use concat function
.otherwise(F.col("validation_errors"))
)

# Rule 4: 'Name' is REQUIRED and should not be null

df_validation_dim_attractions = df_validation_dim_attractions.withColumn(
"validation_errors",
F.when(F.isnull(F.col("Name")), F.concat(F.col("validation_errors"), F.lit("NAME_NULL;"))) # Changed to use concat function
.otherwise(F.col("validation_errors"))
)

df_validation_dim_attractions = df_validation_dim_attractions.withColumn(
  "is_valid",
  F.when(F.length(F.trim(F.col("validation_errors"))) == 0, True).otherwise(False)
 )

df_dim_invalid_attractions = df_validation_dim_attractions.filter(F.col("is_valid") == False).drop("is_valid")
df_dim_gold_attractions = df_validation_dim_attractions.filter(F.col("is_valid") == True).drop("is_valid", "validation_errors")

# Quality Tests fact reviews

In [41]:
df_validation_fact_reviews = df_fact_silver_reviews.withColumn("validation_errors", F.lit(""))

# Rule 1: Username is REQUIRED
df_validation_fact_reviews = df_validation_fact_reviews.withColumn(
    "validation_errors",
    F.when(F.col("Username").isNull(), F.concat(F.col("validation_errors"), F.lit("USERNAME_NULL;")))
     .otherwise(F.col("validation_errors"))
)

# Rule 2: Rating_Review must be between 0 and 50 (adjust if scale is 0–5)
df_validation_fact_reviews = df_validation_fact_reviews.withColumn(
    "validation_errors",
    F.when((F.col("Rating_Review") <= 0) | (F.col("Rating_Review") > 50),
           F.concat(F.col("validation_errors"), F.lit("RATING_REVIEW_INVALID;")))
     .otherwise(F.col("validation_errors"))
)

# Rule 3: Review text is REQUIRED
df_validation_fact_reviews = df_validation_fact_reviews.withColumn(
    "validation_errors",
    F.when(F.col("Review").isNull(), F.concat(F.col("validation_errors"), F.lit("REVIEW_NULL;")))
     .otherwise(F.col("validation_errors"))
)

# Rule 4: Date is REQUIRED
df_validation_fact_reviews = df_validation_fact_reviews.withColumn(
    "validation_errors",
    F.when(F.col("Date_Travel").isNull(), F.concat(F.col("validation_errors"), F.lit("DATE_NULL;")))
     .otherwise(F.col("validation_errors"))
)

# Rule 5: Type_traveler is REQUIRED and must be one of existents categories
categories = ["couples", "families", "alone", "business", "friends"]
df_validation_fact_reviews = df_validation_fact_reviews.withColumn(
    "validation_errors",
    F.when((F.col("Type_traveler").isNull()) | (~F.lower(F.trim(F.col("Type_traveler"))).isin(categories)), F.concat(F.col("validation_errors"), F.lit("TYPE_TRAVELER_INVALID;")))
     .otherwise(F.col("validation_errors"))
)

df_validation_fact_reviews = df_validation_fact_reviews.withColumn(
    "is_valid",
    F.when(F.length(F.trim(F.col("validation_errors"))) == 0, True).otherwise(False)
)

df_fact_invalid_reviews = df_validation_fact_reviews.filter(F.col("is_valid") == False).drop("is_valid")
df_fact_gold_reviews = df_validation_fact_reviews.filter(F.col("is_valid") == True).drop("is_valid", "validation_errors")
# df_fact_invalid_reviews.show()


In [42]:
# spark.stop()

In [56]:
# Export
df_fact_gold_reviews.toPandas().to_csv(config.OUTPUT_PATH_FACT)
df_dim_gold_attractions.toPandas().to_csv(config.OUTPUT_PATH_DIM)